In [1]:
# M5.5 -- Grouping by Hierarchy
# grouping at multiple levels simultaneously
# eg Region to Country to Supplier to Warehouse to SKU
# use .xs() navigate hierarchical results
# combine .groupby() with .unstack()

# import libraries
import pandas as pd
import numpy as np

# create the supply chain spend data set with embedded hierarchy
df = pd.DataFrame({
    "region":    ["North", "North", "North", "North",
                  "South", "South", "South", "South"],
    "warehouse": ["East", "East", "West", "West",
                  "East", "East", "West", "West"],
    "supplier":  ["Acme", "GlobalCo", "Acme", "FastParts",
                  "GlobalCo", "FastParts", "Acme", "GlobalCo"],
    "category":  ["Widget", "Gadget", "Component", "Widget",
                  "Gadget", "Component", "Widget", "Gadget"],
    "spend":     [1500, 2200, 800, 1200,
                  3100, 950, 1800, 2600],
    "units":     [120, 85, 200, 95,
                  140, 300, 110, 175]
})

print(df)


  region warehouse   supplier   category  spend  units
0  North      East       Acme     Widget   1500    120
1  North      East   GlobalCo     Gadget   2200     85
2  North      West       Acme  Component    800    200
3  North      West  FastParts     Widget   1200     95
4  South      East   GlobalCo     Gadget   3100    140
5  South      East  FastParts  Component    950    300
6  South      West       Acme     Widget   1800    110
7  South      West   GlobalCo     Gadget   2600    175


In [3]:
# exercise 2: multi-level groupby
# groupby region and warehouse
# total_spend, total_units, and supplier_count
# fiter for total_spend > 3000
df_hier = (
    df
    .groupby(by=["region", "warehouse"])
    .agg(
        total_spend=("spend", "sum"),
        total_units=("units", "sum"),
        supplier_count=("supplier", "count")
    )
)
print(df_hier[df_hier["total_spend"] > 3000])

                  total_spend  total_units  supplier_count
region warehouse                                          
North  East              3700          205               2
South  East              4050          440               2
       West              4400          285               2


In [ ]:
# exercise 3
# groupby region, warehouse, and supplier
# calculate the total spend
# then unstack() the supplier to columns
# finally, filter the rows where spend > 3,000
# note the NaN's appear after unstacking --> use .fillna(0) to replace NaN's with 0's

df_multi = (
    df
    .groupby(by=["region", "warehouse", "supplier"])
    .agg(
        total_spend=("spend", "sum")
    ) 
    .unstack("supplier")
    .fillna(0)
)

print(df_multi)

                 total_spend                   
supplier                Acme FastParts GlobalCo
region warehouse                               
North  East           1500.0       0.0   2200.0
       West            800.0    1200.0      0.0
South  East              0.0     950.0   3100.0
       West           1800.0       0.0   2600.0


In [8]:
# exercise 4 -- using .xs() cross-section with df_multi

# select only the North Region
print(df_multi.xs(key="North", level="region"))

# select only the East warehouse across all regions
print(df_multi.xs(key="East", level="warehouse"))

# filter the North/East for suppliers with spend > 1000
north_east = df_multi.xs(key="North", level="region")
print(north_east[north_east.gt(1000).any(axis=1)])

          total_spend                   
supplier         Acme FastParts GlobalCo
warehouse                               
East           1500.0       0.0   2200.0
West            800.0    1200.0      0.0
         total_spend                   
supplier        Acme FastParts GlobalCo
region                                 
North         1500.0       0.0   2200.0
South            0.0     950.0   3100.0
          total_spend                   
supplier         Acme FastParts GlobalCo
warehouse                               
East           1500.0       0.0   2200.0
West            800.0    1200.0      0.0
